# Detección de Objetos con YOLO

## Objetivos del ejercicio:
- Entender la diferencia entre clasificación y detección de objetos
- Usar YOLOv8 para detectar múltiples objetos en imágenes
- Ver bounding boxes y confianza en tiempo real
- Comparar YOLO con MobileNet

---

## ¿Qué es YOLO?
- **YOLO** = You Only Look Once
- Detecta **múltiples objetos** en una imagen (no solo clasifica)
- Dibuja **bounding boxes** alrededor de cada objeto
- Muy rápido: puede procesar video en tiempo real
- YOLOv8 es la versión más reciente (2023)

## Clasificación vs Detección

| Clasificación (MobileNet) | Detección (YOLO) |
|---------------------------|------------------|
| "Esta imagen contiene un perro" | "Hay 2 perros en (x,y) y (x2,y2)" |
| Una etiqueta por imagen | Múltiples objetos detectados |
| No indica ubicación | Dibuja bounding boxes |
| Más rápido | Más completo |

## Paso 1: Instalar y cargar librerías

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import requests
from io import BytesIO

print("✅ Librerías cargadas correctamente")

## Paso 2: Cargar el modelo YOLOv8

La primera vez descargará el modelo (~6 MB para YOLOv8n - nano version)

In [ ]:
print("Cargando modelo YOLOv8...")
model = YOLO('yolov8n.pt')
print("✅ Modelo YOLOv8 cargado exitosamente!")
print(f"Clases disponibles: {len(model.names)} categorías")

## Paso 3: Crear funciones auxiliares

**NOTA:** result.plot() de YOLOv8 ya devuelve imágenes en RGB, NO necesitamos conversión BGR→RGB

In [ ]:
def cargar_imagen_desde_url(url):
    """Descarga una imagen desde una URL y la convierte a RGB"""
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return np.array(img)

def cargar_imagen_local(ruta):
    """Carga una imagen desde el disco local y la convierte a RGB"""
    img = Image.open(ruta)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return np.array(img)

def detectar_y_mostrar(img, titulo="Imagen", conf_threshold=0.25):
    """
    Detecta objetos en la imagen y muestra los resultados
    
    Args:
        img: Imagen en formato numpy array RGB
        titulo: Título para mostrar
        conf_threshold: Umbral de confianza mínimo (0-1)
    """
    print(f"\n{'='*60}")
    print(f"Analizando: {titulo}")
    print(f"{'='*60}")
    
    results = model(img, conf=conf_threshold, verbose=False)
    result = results[0]
    
    img_con_boxes = result.plot()
    
    plt.figure(figsize=(14, 8))
    
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Imagen Original', fontsize=14, fontweight='bold')
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_con_boxes)
    plt.axis('off')
    plt.title(f'Detecciones (confianza ≥ {conf_threshold*100:.0f}%)', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    boxes = result.boxes
    num_detecciones = len(boxes)
    
    print(f"\n🎯 Objetos detectados: {num_detecciones}")
    print("\nDetalle de detecciones:")
    print("-" * 60)
    
    if num_detecciones == 0:
        print("⚠️  No se detectaron objetos con el umbral de confianza actual")
        print(f"   Intenta reducir el umbral (actual: {conf_threshold})")
    else:
        for i, box in enumerate(boxes, 1):
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            nombre_clase = model.names[cls]
            
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            ancho = int(x2 - x1)
            alto = int(y2 - y1)
            
            print(f"{i}. {nombre_clase.upper()}")
            print(f"   Confianza: {conf*100:.1f}%")
            print(f"   Posición: ({int(x1)}, {int(y1)}) → ({int(x2)}, {int(y2)})")
            print(f"   Tamaño: {ancho}x{alto} píxeles")
            print()

print("✅ Funciones creadas correctamente")

## Paso 4: ¡Probemos YOLO!

### Ejemplo 1: Calle con múltiples objetos

In [ ]:
url_calle = "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=800"
img_calle = cargar_imagen_desde_url(url_calle)
detectar_y_mostrar(img_calle, "Calle urbana", conf_threshold=0.3)

### Ejemplo 2: Personas en un parque

In [ ]:
url_personas = "https://images.unsplash.com/photo-1511632765486-a01980e01a18?w=800"
img_personas = cargar_imagen_desde_url(url_personas)
detectar_y_mostrar(img_personas, "Personas en el parque", conf_threshold=0.4)

### Ejemplo 3: Animales (perros)

In [ ]:
url_perros = "https://images.unsplash.com/photo-1548199973-03cce0bbc87b?w=800"
img_perros = cargar_imagen_desde_url(url_perros)
detectar_y_mostrar(img_perros, "Perros", conf_threshold=0.3)

### Ejemplo 4: Tráfico (coches, autobuses, personas)

In [ ]:
url_trafico = "https://images.unsplash.com/photo-1449965408869-eaa3f722e40d?w=800"
img_trafico = cargar_imagen_desde_url(url_trafico)
detectar_y_mostrar(img_trafico, "Tráfico urbano", conf_threshold=0.35)

### Ejemplo 5: Comida en una mesa

In [ ]:
url_comida = "https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=800"
img_comida = cargar_imagen_desde_url(url_comida)
detectar_y_mostrar(img_comida, "Mesa con comida", conf_threshold=0.25)

### Ejemplo 6: Deportes (fútbol)

In [ ]:
url_futbol = "https://images.unsplash.com/photo-1579952363873-27f3bade9f55?w=800"
img_futbol = cargar_imagen_desde_url(url_futbol)
detectar_y_mostrar(img_futbol, "Partido de fútbol", conf_threshold=0.3)

## Experimento: Ajustar el umbral de confianza

Prueba la misma imagen con diferentes umbrales para ver cómo afecta las detecciones

In [ ]:
print("Comparando diferentes umbrales de confianza:\n")

url_test = "https://images.unsplash.com/photo-1449824913935-59a10b8d2000?w=800"
img_test = cargar_imagen_desde_url(url_test)

for umbral in [0.1, 0.3, 0.5, 0.7]:
    print(f"\n{'='*60}")
    print(f"Umbral de confianza: {umbral*100:.0f}%")
    print(f"{'='*60}")
    detectar_y_mostrar(img_test, f"Umbral {umbral*100:.0f}%", conf_threshold=umbral)

## Ejercicio: Prueba con tus propias imágenes

### Opción 1: Desde URL

In [ ]:
mi_url = "PEGA_AQUI_TU_URL"
mi_imagen = cargar_imagen_desde_url(mi_url)
detectar_y_mostrar(mi_imagen, "Mi imagen", conf_threshold=0.3)

### Opción 2: Desde archivo local

In [ ]:
ruta_local = "imagenes/mi_foto.jpg"
mi_imagen_local = cargar_imagen_local(ruta_local)
detectar_y_mostrar(mi_imagen_local, "Mi foto local", conf_threshold=0.3)

## Ver todas las clases que YOLO puede detectar

In [ ]:
print(f"YOLO puede detectar {len(model.names)} clases diferentes:\n")
print("="*60)

for idx, nombre in model.names.items():
    print(f"{idx:2d}. {nombre}")

print("\n" + "="*60)
print("\nCategorías comunes:")
print("- Personas y animales: person, dog, cat, bird, horse, etc.")
print("- Vehículos: car, truck, bus, motorcycle, bicycle, etc.")
print("- Objetos cotidianos: chair, bottle, cup, laptop, phone, etc.")
print("- Deportes: sports ball, tennis racket, skateboard, etc.")
print("- Comida: pizza, donut, cake, apple, sandwich, etc.")

## Comparación: MobileNet vs YOLO

| Característica | MobileNet (Clasificación) | YOLO (Detección) |
|----------------|---------------------------|------------------|
| **Tarea** | Clasificar imagen completa | Detectar múltiples objetos |
| **Salida** | 1 etiqueta + confianza | N bounding boxes + etiquetas |
| **Ubicación** | ❌ No indica dónde está | ✅ Coordenadas exactas |
| **Múltiples objetos** | ❌ Solo objeto principal | ✅ Todos los objetos |
| **Velocidad** | Muy rápido (~10ms) | Rápido (~30ms) |
| **Clases** | 1000 (ImageNet) | 80 (COCO dataset) |
| **Uso típico** | Organizar fotos, búsqueda | Vigilancia, conducción autónoma |

### ¿Cuándo usar cada uno?

**Usa MobileNet cuando:**
- Solo necesitas saber QUÉ hay en la imagen
- Clasificación simple (ej: "¿es un gato o un perro?")
- Necesitas máxima velocidad
- Trabajas con muchas categorías (1000)

**Usa YOLO cuando:**
- Necesitas saber DÓNDE están los objetos
- Hay múltiples objetos en la imagen
- Necesitas contar objetos
- Aplicaciones de seguridad o análisis espacial

## Conceptos clave aprendidos

**Detección de objetos**: Identificar y localizar múltiples objetos en una imagen

**Bounding box**: Rectángulo que encierra un objeto detectado

**Umbral de confianza**: Nivel mínimo de certeza para considerar una detección válida

**COCO dataset**: Dataset de 80 categorías usado para entrenar YOLO

**Tiempo real**: YOLO puede procesar 30+ imágenes por segundo

---

## Aplicaciones reales de YOLO

1. **Conducción autónoma**: Detectar peatones, coches, señales
2. **Vigilancia**: Contar personas, detectar intrusos
3. **Retail**: Análisis de inventario, detección de productos
4. **Deportes**: Seguimiento de jugadores y balón
5. **Manufactura**: Control de calidad, detección de defectos
6. **Agricultura**: Contar animales, detectar plagas
7. **Medicina**: Detectar anomalías en imágenes médicas

---

## Recursos adicionales

- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com/)
- [COCO Dataset](https://cocodataset.org/)
- [YOLO Paper Original](https://arxiv.org/abs/1506.02640)
- [Roboflow - Entrenar YOLO custom](https://roboflow.com/)

---

## Desafíos para practicar

1. **Contador de personas**: Detecta cuántas personas hay en una foto grupal
2. **Análisis de tráfico**: Cuenta coches, motos y autobuses en una calle
3. **Detector de mascotas**: Identifica perros y gatos en fotos
4. **Inventario visual**: Detecta objetos en tu escritorio
5. **Umbral óptimo**: Encuentra el mejor umbral para diferentes escenarios

**¡Experimenta y diviértete! 🚀**